In [ ]:
"""
Using demo code to experiment the model aneuraz/awesome-align-with-co from Hugging Face.
The task is to find word alignments by using multilingual BERT(mBERT).
"""

In [ ]:
from transformers import AutoModel, AutoTokenizer
import itertools
import torch

In [ ]:
# load model
model = AutoModel.from_pretrained("aneuraz/awesome-align-with-co")
tokenizer = AutoTokenizer.from_pretrained("aneuraz/awesome-align-with-co")

# model parameters
align_layer = 8
threshold = 1e-3


config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

Some weights of BertModel were not initialized from the model checkpoint at aneuraz/awesome-align-with-co and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
# define inputs

tgt = ['科学家', '将', '习惯', '定义', '为', '定期', '进行', '的', '行为', '，', '这些', '行为', '是',
       '在', '潜意识', '中', '被',
       '某些', '特定', '环境', '激发', '的', '，', '无论是', '具体', '地点', '、', '特定', '时间', '，',
       '还是', '某种', '情绪', '状态', '。']
src = ['Scientists', 'define', 'habits', 'as', 'behaviors', 'that', 'are', 'performed', 'regularly',
       ',',
       'and', 'cued', 'subconsciously', 'in', 'response', 'to', 'certain', 'environments', ',',
       'whether', 'it', 'be', 'a', 'location', ',', 'time', 'of', 'day', ',', 'or', 'even', 'an',
       'emotional', 'state', '.']



In [ ]:
# pre-processing
# sent_src, sent_tgt = src.strip().split(), tgt.strip().split()
token_src, token_tgt = [tokenizer.tokenize(word) for word in src], [tokenizer.tokenize(word) for word in tgt]
wid_src, wid_tgt = [tokenizer.convert_tokens_to_ids(x) for x in token_src], [tokenizer.convert_tokens_to_ids(x) for x in token_tgt]
ids_src, ids_tgt = tokenizer.prepare_for_model(list(itertools.chain(*wid_src)), return_tensors='pt', model_max_length=tokenizer.model_max_length, truncation=True)['input_ids'], tokenizer.prepare_for_model(list(itertools.chain(*wid_tgt)), return_tensors='pt', truncation=True, model_max_length=tokenizer.model_max_length)['input_ids']
sub2word_map_src = []
for i, word_list in enumerate(token_src):
  sub2word_map_src += [i for x in word_list]
sub2word_map_tgt = []
for i, word_list in enumerate(token_tgt):
  sub2word_map_tgt += [i for x in word_list]


You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


In [ ]:
# alignment
align_layer = 8
threshold = 1e-3
model.eval()
with torch.no_grad():
  out_src = model(ids_src.unsqueeze(0), output_hidden_states=True)[2][align_layer][0, 1:-1]
  out_tgt = model(ids_tgt.unsqueeze(0), output_hidden_states=True)[2][align_layer][0, 1:-1]

  dot_prod = torch.matmul(out_src, out_tgt.transpose(-1, -2))

  softmax_srctgt = torch.nn.Softmax(dim=-1)(dot_prod)
  softmax_tgtsrc = torch.nn.Softmax(dim=-2)(dot_prod)

  softmax_inter = (softmax_srctgt > threshold)*(softmax_tgtsrc > threshold)

align_subwords = torch.nonzero(softmax_inter, as_tuple=False)
align_words = set()
for i, j in align_subwords:
  align_words.add( (sub2word_map_src[i], sub2word_map_tgt[j]) )

print(align_words)

{(6, 12), (3, 4), (6, 21), (26, 27), (5, 7), (22, 23), (23, 25), (2, 2), (34, 34), (1, 3), (27, 28), (11, 20), (25, 28), (24, 26), (18, 22), (4, 8), (8, 5), (31, 31), (19, 23), (28, 29), (13, 13), (32, 32), (7, 6), (16, 18), (21, 23), (29, 30), (0, 0), (9, 9), (14, 20), (17, 19), (33, 33)}


In [ ]:
for align in sorted(align_words):
  print(src[align[0]], tgt[align[1]])

Scientists 科学家
define 定义
habits 习惯
as 为
behaviors 行为
that 的
are 是
are 的
performed 进行
regularly 定期
, ，
cued 激发
in 在
response 激发
certain 特定
environments 环境
, ，
whether 无论是
be 无论是
a 无论是
location 地点
, 、
time 时间
of 特定
day 时间
, ，
or 还是
an 某种
emotional 情绪
state 状态
. 。
